# Local RAG with Gemma 4 + Ollama

This notebook demonstrates how to build a **fully local** RAG (Retrieval-Augmented Generation) pipeline using:
- **Ollama** for the LLM (runs locally, no API keys needed)
- **HuggingFace Embeddings** for document embedding (sentence-transformers)
- **LlamaIndex** for orchestrating the RAG pipeline

## RAG Pipeline Steps

1. **Load** - Read the PDF document
2. **Chunk** - Split into manageable pieces
3. **Embed** - Convert chunks to vector representations
4. **Index** - Store vectors for efficient retrieval
5. **Query** - Retrieve relevant chunks and generate answers

## Prerequisites

1. **Install Ollama**: Download from [ollama.ai](https://ollama.ai)
2. **Pull a model**: Run `ollama pull gemma4` in your terminal
3. **Install Python packages**: Run the cell below

In [8]:
# Install required packages
# IMPORTANT: use %pip (not !pip) so packages land in THIS kernel's env.
# After running this cell the first time, restart the kernel before continuing —
# otherwise `SimpleDirectoryReader` will have already imported without the PDF
# reader plugin and will silently fall back to reading PDFs as raw bytes.
%pip install -q llama-index-core llama-index-llms-ollama llama-index-embeddings-huggingface
%pip install -q llama-index-readers-file pypdf sentence-transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Step 1: Configure the LLM and Embedding Model

We'll use:
- **Ollama** with `gemma4` for text generation (runs on localhost:11434)
- **BGE-small** from HuggingFace for embeddings (384-dim, fast & accurate)

In [7]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure the LLM - Ollama runs locally on port 11434
Settings.llm = Ollama(
    model="gemma4",           # Use gemma4 (course default model)
    request_timeout=120.0,       # Timeout for generation
    temperature=0.1,             # Low temperature for factual responses
)

# Configure the embedding model - runs locally via sentence-transformers
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",  # 384-dim, ~130MB
)

print("LLM and Embedding model configured!")

2026-05-12 15:17:11,437 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 15:17:11,445 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-05-12 15:17:11,580 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 15:17:11,588 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-12 15:17:11,589 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-05-12 15:17:11,756 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-12 15:17:12,670 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-12 15:17:12,806 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-12 15:17:12,947 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-12 15:17:13,086 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-12 15:17:13,231 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 15:17:13,239 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

LLM and Embedding model configured!


## Step 2: Load the PDF Document

LlamaIndex's `SimpleDirectoryReader` handles PDF parsing automatically.
Each page becomes a separate Document object with metadata.

In [8]:
from llama_index.core import SimpleDirectoryReader

PDF_DIR = "/Users/greatmaster/Desktop/projects/oreilly-live-trainings/llama2_oreilly_live_training/notebooks/assets-resources/pdf-test"

documents = SimpleDirectoryReader(input_dir=PDF_DIR).load_data()
print(f"Loaded {len(documents)} pages from the PDF")


def validate_loaded_documents(documents) -> None:
    # Fail fast if SimpleDirectoryReader silently fell back to reading the PDF
    # as raw bytes (happens when llama-index-readers-file / pypdf aren't in the kernel).
    if len(documents) == 1 and documents[0].text.lstrip().startswith("%PDF-"):
        raise AssertionError(
            "PDF loaded as raw bytes — the PDF reader plugin is missing.\n"
            "Fix: %pip install llama-index-readers-file pypdf, then restart the kernel."
        )
    assert len(documents) > 1, (
        f"Expected a multi-page PDF, got {len(documents)} document(s). "
        "Check the input_dir and the PDF reader plugin."
    )


validate_loaded_documents(documents)
print("Load looks healthy.")

Loaded 11 pages from the PDF
Load looks healthy.


## Step 3: Create the Vector Index

This step:
1. Chunks the documents (default: 1024 tokens with 20 overlap)
2. Generates embeddings for each chunk
3. Stores them in an in-memory vector store

For production, you'd use a persistent vector store like ChromaDB or FAISS.

In [9]:
from llama_index.core import VectorStoreIndex

# Create the index - this embeds all chunks
index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True,  # Show embedding progress
)

print("\nIndex created successfully!") 

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/12 [00:00<?, ?it/s]


Index created successfully!


## Step 4: Create the Query Engine

The query engine combines:
- **Retriever**: Finds relevant chunks using vector similarity
- **Response Synthesizer**: Generates answers using the LLM

`similarity_top_k=3` means we retrieve the 3 most relevant chunks.

In [10]:
# Create query engine with top-3 retrieval
query_engine = index.as_query_engine(
    similarity_top_k=3,  # Number of chunks to retrieve
)

print("Query engine ready!")

2026-05-12 15:17:50,104 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


Query engine ready!


## Step 5: Query the Documents

Now we can ask questions about the paper! The RAG pipeline will:
1. Embed your question
2. Find similar chunks in the index
3. Send chunks + question to the LLM
4. Return the generated answer

In [11]:
# Ask a question about the paper
response = query_engine.query("What is the core thesis of this attention paper??")

print("Answer:")
print(response.response)

2026-05-12 15:18:45,546 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Answer:
The mechanism involves calculating attention using the formula $\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$.

Key aspects of the attention function include:
*   **Dot-Product Attention:** This is a common method, which is faster and more space-efficient than additive attention because it can be implemented using highly optimized matrix multiplication code.
*   **Scaling:** To prevent the dot products from becoming too large in magnitude, which could cause the softmax function to have extremely small gradients, the dot products are scaled by $\frac{1}{\sqrt{d_k}}$.
*   **Multi-Head Attention:** This technique enhances the model by performing the attention function in parallel multiple times ($h$ times). It achieves this by linearly projecting the queries, keys, and values with different learned linear projections. This process allows the model to jointly attend to information from different representation subspaces at different positions.


In [12]:
# Let's try another question
response = query_engine.query("What is self-attention and how does it work?Answer in 5 bullet short points")

print("Answer:")
print(response.response)

2026-05-12 15:19:32,440 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Answer:
*   Self-attention enables every position within a sequence to attend to all other positions in that same sequence.
*   The mechanism uses the same source for the queries, keys, and values, meaning all three components are derived from the output of the previous layer.
*   It is beneficial for learning long-range dependencies because it connects all positions with a constant number of sequential operations.
*   Compared to recurrent layers, it is computationally faster when the sequence length is smaller than the representation dimensionality.
*   When used in the decoder, a masking process is applied during self-attention to prevent information flow from future positions, thereby preserving the auto-regressive property.


## Inspecting Retrieved Sources

One advantage of RAG is transparency - we can see which chunks were used to generate the answer.

In [13]:
# Inspect the source nodes (retrieved chunks)
print(f"Number of source chunks: {len(response.source_nodes)}\n")

for i, node in enumerate(response.source_nodes):
    print(f"--- Source {i+1} (score: {node.score:.3f}) ---")
    print(f"{node.text[:300]}...\n")

Number of source chunks: 3

--- Source 1 (score: 0.685) ---
MultiHead(Q,K,V ) = Concat(head 1,..., headh)W O
where headi = Attention(QW Q
i ,KW K
i ,VW V
i )
Where the projections are parameter matricesW Q
i ∈ Rdmodel×dk,W K
i ∈ Rdmodel×dk,W V
i ∈ Rdmodel×dv
andW O∈ Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of th...

--- Source 2 (score: 0.672) ---
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by√dk, and apply a softmax function to obtain the weights on the
values.
In practi...

--- Source 3 (score: 0.650) ---
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
size of convolutions andr the size of the neighborhood in restricted 

## (Optional) Persist the Index

Save the index to disk so you don't need to re-embed documents each time.

In [14]:
# Save the index to disk
index.storage_context.persist(persist_dir="./storage/attention_paper")
print("Index saved to ./storage/attention_paper/")

Index saved to ./storage/attention_paper/


In [ ]:
# Uncomment to load:

# To load the index later:
# from llama_index.core import StorageContext, load_index_from_storage


# storage_context = StorageContext.from_defaults(persist_dir="./storage/attention_paper")
# loaded_index = load_index_from_storage(storage_context)
# query_engine = loaded_index.as_query_engine(similarity_top_k=3)

## Summary

You've built a complete local RAG pipeline! Key components:

| Component | Tool | Why |
|-----------|------|-----|
| LLM | Ollama (gemma4) | Local, no API keys, easy model management |
| Embeddings | HuggingFace (bge-small) | Fast, accurate, runs locally |
| Orchestration | LlamaIndex | Handles chunking, indexing, retrieval |

### Next Steps
- Try different models: `ollama pull mistral` or `ollama pull phi3`
- Use persistent storage: ChromaDB, FAISS
- Experiment with chunk sizes via `Settings.chunk_size`
- Add hybrid search (keyword + semantic)